In [0]:
# Databricks Notebook — 03_gold_aggregations.py
# Layer   : Gold
# Purpose : Produce four Delta Lake output tables from Silver joined data:
#             1. fraud_metrics      — aggregated fraud stats by entity
#             2. reporting_table    — dashboard-ready flat table
#             3. alerts             — high-risk flagged records
#             4. ml_features        — model-ready feature table
# Run order: 3 of 3  (requires 02_silver_transform to have run)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------------------------------------------------
# 0. CONFIGURATION  ← must match 01_bronze_reader.py and 02_silver_transform.py
# ---------------------------------------------------------------------------
STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT")
STORAGE_KEY     = os.getenv("STORAGE_KEY")
CONTAINER  = os.getenv("CONTAINER_NAME")

GOLD_BASE  = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/fraud_gold"

FRAUD_METRICS_DELTA_PATH  = f"{GOLD_BASE}/delta/fraud_metrics/"
REPORTING_DELTA_PATH      = f"{GOLD_BASE}/delta/reporting_table/"
ALERTS_DELTA_PATH         = f"{GOLD_BASE}/delta/alerts/"
ML_FEATURES_DELTA_PATH    = f"{GOLD_BASE}/delta/ml_features/"

DATABASE = "fraud_lakehouse"   # must match Bronze and Silver

# Risk threshold — transactions with fraud-score above this are flagged
ALERT_FRAUD_SCORE_THRESHOLD = 0.7

# ---------------------------------------------------------------------------
# 1. RE-APPLY ADLS Gen2 AUTHENTICATION + DELTA OPTIMIZATIONS
# ---------------------------------------------------------------------------
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",   "true")
spark.conf.set("spark.sql.adaptive.enabled",                   "true")

spark.sql(f"USE {DATABASE}")

print("✅  ADLS Gen2 authentication configured.")

In [0]:
# ---------------------------------------------------------------------------
# 2. READ SILVER JOINED DELTA TABLE FROM METASTORE
#    spark.table() reads directly from the Delta metastore —
#    no paths, no temp views, no session state dependency.
#    Cached once and reused across all four Gold aggregations.
# ---------------------------------------------------------------------------
silver = spark.table(f"{DATABASE}.silver_joined").cache()

assert silver.count() > 0, "Gold abort: silver_joined Delta table is empty — check Silver notebook!"

print(f"✅  silver_joined : {silver.count():,} rows loaded from Delta.")


In [0]:
# ---------------------------------------------------------------------------
# 3. GOLD TABLE 1 — FRAUD AGGREGATION METRICS
#    Entity-level summary: card, product category, device type
# ---------------------------------------------------------------------------

# ── 3a. By card1 (card number prefix)
card_metrics = (
    silver
    .groupBy("card1")
    .agg(
        F.count("TransactionID").alias("total_transactions"),
        F.sum("isFraud").alias("total_fraud"),
        F.avg("isFraud").alias("fraud_rate"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.avg("TransactionAmt").alias("avg_amount"),
        F.max("TransactionAmt").alias("max_amount"),
    )
    .withColumn("entity_type", F.lit("card1"))
    .withColumnRenamed("card1", "entity_value")
    .withColumn("entity_value", F.col("entity_value").cast("string"))
)

# ── 3b. By ProductCD
product_metrics = (
    silver
    .groupBy("ProductCD")
    .agg(
        F.count("TransactionID").alias("total_transactions"),
        F.sum("isFraud").alias("total_fraud"),
        F.avg("isFraud").alias("fraud_rate"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.avg("TransactionAmt").alias("avg_amount"),
        F.max("TransactionAmt").alias("max_amount"),
    )
    .withColumn("entity_type", F.lit("ProductCD"))
    .withColumnRenamed("ProductCD", "entity_value")
    .withColumn("entity_value", F.col("entity_value").cast("string"))
)

# ── 3c. By DeviceType
device_metrics = (
    silver
    .groupBy("DeviceType")
    .agg(
        F.count("TransactionID").alias("total_transactions"),
        F.sum("isFraud").alias("total_fraud"),
        F.avg("isFraud").alias("fraud_rate"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.avg("TransactionAmt").alias("avg_amount"),
        F.max("TransactionAmt").alias("max_amount"),
    )
    .withColumn("entity_type", F.lit("DeviceType"))
    .withColumnRenamed("DeviceType", "entity_value")
    .withColumn("entity_value", F.col("entity_value").cast("string"))
)

fraud_metrics = card_metrics.unionByName(product_metrics).unionByName(device_metrics)

(
    fraud_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", FRAUD_METRICS_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.gold_fraud_metrics")
)
print(f"✅  gold_fraud_metrics written  ({fraud_metrics.count():,} rows)")

In [0]:
# ---------------------------------------------------------------------------
# 4. GOLD TABLE 2 — REPORTING TABLE  (dashboard-ready flat table)
# ---------------------------------------------------------------------------

# Rolling 7-day fraud count per card.
# rangeBetween() requires a numeric order key — TransactionDT (LongType
# seconds-offset from Silver) is included in the select, used for the window,
# then dropped from the final output.
card_window = (
    Window
    .partitionBy("card1")
    .orderBy("TransactionDT")
    .rangeBetween(-7 * 86400, 0)        # 7 days in seconds
)

reporting_table = (
    silver
    .select(
        "TransactionID",
        "TransactionDT",                 # kept for window ordering — dropped after
        "TransactionTimestamp",
        "TransactionAmt",
        "ProductCD",
        "card1",
        "card4",                         # card brand (Visa, MC, etc.)
        "card6",                         # debit / credit
        "addr1",
        "addr2",
        "P_emaildomain",
        "R_emaildomain",
        "DeviceType",
        "DeviceInfo",
        "isFraud",
    )
    .withColumn("rolling_7d_fraud_count", F.sum("isFraud").over(card_window))
    .drop("TransactionDT")
    .withColumn(
        "amt_band",
        F.when(F.col("TransactionAmt") < 50,   "low")
         .when(F.col("TransactionAmt") < 500,  "medium")
         .when(F.col("TransactionAmt") < 5000, "high")
         .otherwise("very_high")
    )
    .withColumn(
        "fraud_label",
        F.when(F.col("isFraud") == 1, "fraud").otherwise("legitimate")
    )
)

(
    reporting_table
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ProductCD")            # Synapse queries commonly filter by ProductCD
    .option("path", REPORTING_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.gold_reporting_table")
)
print(f"✅  gold_reporting_table written ({reporting_table.count():,} rows)")

In [0]:

# ---------------------------------------------------------------------------
# 5. GOLD TABLE 3 — ALERTS  (high-risk flagged records)
# ---------------------------------------------------------------------------
alerts_scored = (
    silver
    .withColumn(
        "fraud_score",
        (
            F.col("isFraud").cast("double") * 0.5
            + F.when(F.col("TransactionAmt") > 1000, 0.2).otherwise(0.0)
            + F.when(F.col("addr2") != 87.0, 0.15).otherwise(0.0)
            + F.when(F.col("DeviceType").isNull(), 0.1).otherwise(0.0)
            + F.when(F.col("P_emaildomain") != F.col("R_emaildomain"), 0.05).otherwise(0.0)
        )
    )
)

alerts = (
    alerts_scored
    .filter(F.col("fraud_score") >= ALERT_FRAUD_SCORE_THRESHOLD)
    .select(
        "TransactionID",
        "TransactionTimestamp",
        "TransactionAmt",
        "card1",
        "addr1",
        "addr2",
        "DeviceType",
        "DeviceInfo",
        "P_emaildomain",
        "R_emaildomain",
        "isFraud",
        "fraud_score",
    )
    .orderBy(F.col("fraud_score").desc())
)

(
    alerts
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", ALERTS_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.gold_alerts")
)
print(f"✅  gold_alerts written          ({alerts.count():,} flagged records)")

In [0]:
# ---------------------------------------------------------------------------
# 6. GOLD TABLE 4 — ML FEATURE TABLE
# ---------------------------------------------------------------------------
card_hist = (
    silver
    .groupBy("card1")
    .agg(
        F.count("TransactionID").alias("card_txn_count"),
        F.avg("TransactionAmt").alias("card_avg_amt"),
        F.stddev("TransactionAmt").alias("card_std_amt"),
        F.avg("isFraud").alias("card_historical_fraud_rate"),
    )
)

ml_features = (
    silver
    .join(card_hist, on="card1", how="left")
    .select(
        "TransactionID",
        "isFraud",
        "TransactionAmt",
        "TransactionDT",
        "card1", "card2",
        "addr1", "addr2",
        "dist1", "dist2",
        "card_txn_count",
        "card_avg_amt",
        "card_std_amt",
        "card_historical_fraud_rate",
        F.when(F.col("card_avg_amt") > 0,
               F.col("TransactionAmt") / F.col("card_avg_amt")
        ).otherwise(F.lit(1.0)).alias("amt_vs_card_avg_ratio"),
        F.when(F.col("DeviceType").isNull(), 1).otherwise(0).alias("is_unknown_device"),
        F.when(F.col("P_emaildomain") != F.col("R_emaildomain"), 1).otherwise(0).alias("email_domain_mismatch"),
        F.when(F.col("addr2") != 87.0, 1).otherwise(0).alias("is_foreign_billing"),
        *[F.col(f"C{i}") for i in range(1, 15)],
        *[F.col(f"D{i}") for i in range(1, 16)],
        *[F.col(f"V{i}") for i in [1, 2, 3, 4, 12, 13, 14, 15, 70, 95, 96, 130, 166]],
        *[F.col(f"M{i}") for i in range(1, 10)],
    )
    .fillna(0)
)

(
    ml_features
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", ML_FEATURES_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.gold_ml_features")
)
print(f"✅  gold_ml_features written     ({ml_features.count():,} rows)")


In [0]:
# ---------------------------------------------------------------------------
# 7. OPTIMIZE GOLD DELTA TABLES
# ---------------------------------------------------------------------------
spark.sql(f"OPTIMIZE {DATABASE}.gold_fraud_metrics")
spark.sql(f"OPTIMIZE {DATABASE}.gold_reporting_table ZORDER BY (card1, TransactionTimestamp)")
spark.sql(f"OPTIMIZE {DATABASE}.gold_alerts          ZORDER BY (fraud_score)")
spark.sql(f"OPTIMIZE {DATABASE}.gold_ml_features     ZORDER BY (TransactionID)")

print("\n✅  All Gold Delta tables optimized.")

In [0]:
# ---------------------------------------------------------------------------
# 8. SUMMARY
# ---------------------------------------------------------------------------
print("\n══════════════════════════════════════════════════════════")
print(f"  Gold Layer Complete — database: {DATABASE}")
print(f"  ├── gold_fraud_metrics   : {FRAUD_METRICS_DELTA_PATH}")
print(f"  ├── gold_reporting_table : {REPORTING_DELTA_PATH}")
print(f"  ├── gold_alerts          : {ALERTS_DELTA_PATH}")
print(f"  └── gold_ml_features     : {ML_FEATURES_DELTA_PATH}")
print("══════════════════════════════════════════════════════════")
print("▶  Gold tables are ready for Synapse Analytics consumption.")